# Missing-grants acquisition roadmap (regenerable)

Rebuilds the per-funder missing-grants roadmap from the production verdict
table `openalex.awards.award_id_verdicts` — the living version of
oxjobs `todo/crossref-award-validation/MISSING-GRANTS-ROADMAP.md`
(Kyle's ask #5, 2026-07-29: "unmatched crossref-provenance rows = roadmap
for acquiring missing grants").

**missing_grammar_pass** = publisher-deposited ids that fit the funder's
real grant-number format but are absent from the registry we ingested —
overwhelmingly real grants our registry copy lacks (measured purity at
NSFC: 29/30 of a random sample carry at least one verifiable NSFC id).

The verdict table is recomputed on the registry-ingest cadence, so
re-running this notebook refreshes the roadmap at any time.


In [0]:
%sql
-- Rebuild the per-funder missing-grants roadmap from the production verdict
-- table. Methodology (must stay in sync with the published roadmap doc in
-- oxjobs todo/crossref-award-validation/MISSING-GRANTS-ROADMAP.md):
--   matched  = confirmed + confirmed_weak
--   missing  = plausible  (grammar-pass, no registry match — the roadmap column)
--   garbage  = positively-classified junk
-- Extra columns (confirmed_ambiguous / foreign_scheme / unscored) are
-- transparency-only and not part of the published headline numbers.
--
-- Target: dev while staged. Prod home (e.g. openalex.awards.missing_grants_roadmap)
-- is a one-line flip at landing — Rohan's call.
CREATE OR REPLACE TABLE openalex_dev.rohan_lab.missing_grants_roadmap_prodready AS
WITH names AS (
  SELECT id, min(name) AS funder_name
  FROM openalex.common.funders
  GROUP BY id
),
per_funder AS (
  SELECT
    funder_id,
    count(DISTINCT funder_award_id)                                                      AS deposited_ids,
    count(DISTINCT CASE WHEN verdict IN ('confirmed','confirmed_weak')
                        THEN funder_award_id END)                                        AS registry_matched,
    count(DISTINCT CASE WHEN verdict = 'plausible'           THEN funder_award_id END)   AS missing_grammar_pass,
    count(DISTINCT CASE WHEN verdict = 'garbage'             THEN funder_award_id END)   AS garbage,
    count(DISTINCT CASE WHEN verdict = 'confirmed_ambiguous' THEN funder_award_id END)   AS confirmed_ambiguous,
    count(DISTINCT CASE WHEN verdict = 'foreign_scheme'      THEN funder_award_id END)   AS foreign_scheme,
    count(DISTINCT CASE WHEN verdict = 'unscored'            THEN funder_award_id END)   AS unscored
  FROM openalex.awards.award_id_verdicts
  GROUP BY funder_id
)
SELECT
  p.funder_id,
  n.funder_name,
  p.deposited_ids,
  p.registry_matched,
  p.missing_grammar_pass,
  p.garbage,
  p.confirmed_ambiguous,
  p.foreign_scheme,
  p.unscored,
  round(p.registry_matched / nullif(p.registry_matched + p.missing_grammar_pass, 0), 3)
    AS registry_coverage_of_grammar_pass,
  current_timestamp() AS generated_at
FROM per_funder p
LEFT JOIN names n ON n.id = concat('https://openalex.org/F', cast(p.funder_id AS string))
-- configured funders only: at least one scored verdict
WHERE p.registry_matched + p.missing_grammar_pass + p.garbage
      + p.confirmed_ambiguous + p.foreign_scheme > 0


In [0]:
%sql
-- Top of the roadmap, as published in the oxjobs doc
SELECT funder_name, deposited_ids, registry_matched, missing_grammar_pass, garbage
FROM openalex_dev.rohan_lab.missing_grants_roadmap_prodready
ORDER BY missing_grammar_pass DESC
LIMIT 25
